In [2]:
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, avg, count, when, sum
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from credentials import params
session = Session.builder.configs(params).create()

In [4]:
df_snowpark = session.table("HOUSING_PRICE_PROJECT.STAGING_LAYER.RAW_DATA")

In [5]:
df = session.sql("SELECT * FROM HOUSING_PRICE_PROJECT.STAGING_LAYER.RAW_DATA").to_pandas()

In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 31 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   property_id                 35000 non-null  str    
 1   city                        35000 non-null  str    
 2   locality                    35000 non-null  str    
 3   locality_tier               35000 non-null  str    
 4   property_type               35000 non-null  str    
 5   bhk                         35000 non-null  int8   
 6   bathrooms                   35000 non-null  int8   
 7   balconies                   35000 non-null  int8   
 8   built_up_area               35000 non-null  int16  
 9   carpet_area                 35000 non-null  int16  
 10  floor_number                35000 non-null  int8   
 11  total_floors                35000 non-null  int8   
 12  floor_category              35000 non-null  str    
 13  facing                      35000 non-null

In [6]:
df.describe()

,bhk,bathrooms,balconies,built_up_area,carpet_area,floor_number,total_floors,property_age,parking_spaces,security_score,...,swimming_pool,power_backup,lift_available,maintenance_fee_monthly,distance_to_city_center_km,distance_to_metro_km,nearby_schools,nearby_hospitals,price_in_lakhs,FILE_LAST_MODIFIED
count,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.00000,35000.000000,35000.000000,35000.000000,35000.000000,...,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000
mean,3.151829,3.365343,1.681971,1573.101686,1321.418886,9.33900,14.443086,6.504486,1.919514,5.999220,...,0.299029,0.684657,0.840686,3981.710486,11.541657,2.509520,3.120657,1.906086,166.772213,2026-07-27 18:46:40
min,1.000000,1.000000,0.000000,300.000000,240.000000,0.00000,1.000000,0.000000,0.000000,0.100000,...,0.000000,0.000000,0.000000,500.000000,4.000000,0.100000,0.000000,0.000000,10.000000,2026-07-27 18:46:40
25%,2.000000,2.000000,1.000000,1046.750000,878.000000,3.00000,8.000000,2.000000,1.000000,4.900000,...,0.000000,0.000000,1.000000,2682.000000,6.800000,0.700000,2.000000,1.000000,86.127500,2026-07-27 18:46:40
50%,3.000000,3.000000,2.000000,1587.000000,1330.000000,7.00000,12.000000,4.000000,2.000000,5.900000,...,0.000000,1.000000,1.000000,3667.000000,11.000000,1.800000,3.000000,2.000000,148.000000,2026-07-27 18:46:40
75%,4.000000,4.000000,3.000000,2131.000000,1784.000000,14.00000,18.000000,9.000000,3.000000,7.100000,...,1.000000,1.000000,1.000000,5180.000000,14.300000,3.500000,4.000000,3.000000,226.790000,2026-07-27 18:46:40
max,5.000000,6.000000,4.000000,3056.000000,2666.000000,39.00000,45.000000,40.000000,5.000000,10.000000,...,1.000000,1.000000,1.000000,8859.000000,45.000000,15.000000,10.000000,8.000000,737.590000,2026-07-27 18:46:40
std,1.214986,1.329136,1.232055,669.493721,563.581272,8.70655,8.817221,6.880724,1.451722,1.601971,...,0.457839,0.464659,0.365974,1637.896999,5.452530,2.477522,1.898308,1.467227,104.714704,NaN


# Correlation Matrix

In [ ]:
numerical_data = df.select_dtypes(include=['number'])
corr_matr = numerical_data.corr(method='pearson')

In [ ]:
plt.figure(figsize=(20,16))
sns.heatmap(corr_matr, annot=True, cmap='coolwarm')

In [ ]:
df.head()

In [ ]:
df['property_id'].isnull().sum()

In [ ]:
df.isnull().sum()

# Something with Snowpark

In [70]:
sourcefile_list_raw = df_snowpark.select(col('SOURCE_FILE')).distinct().collect()
sourcefile_list = []

for i in sourcefile_list_raw:
    for j in i:
        sourcefile_list.append(j)


sourcefile_list

['HousingPricePredictionDataset_train.csv']

In [71]:
sourcefile_dqr_list = []
table_exist = False

try:
    dqr_df = session.table("HOUSING_PRICE_PROJECT.STAGING_LAYER.DATA_QUALITY_REPORT")
    sourcefile_dqr_raw = dqr_df.select(col('SOURCE_FILE')).distinct().collect()
    table_exist = True
    
    for i in sourcefile_dqr_raw:
        for j in i:
            sourcefile_dqr_list.append(j)
except:
    pass

print(sourcefile_dqr_list)
print(table_exist)

['HousingPricePredictionDataset_train.csv']
True


In [72]:
from itertools import product

tab_name = "TEST_TAB2"


for sc in sourcefile_list:

    if sc in sourcefile_dqr_list:
        continue
    
    
    df_filtered = df_snowpark.filter(
        (col('SOURCE_FILE') == sc)
    )

    schema = ['SOURCE_FILE', 'CHECK_TYPE', 'OTHER']
    line = ["'" + sc + "'"]    
    
    cols = df_filtered.columns
    cols.remove('SOURCE_FILE')

    cols

    # CREATE TAB HEADER (schema) AND ROW WITH MISSING VALUE (line) ****************************************************
    line, schema = null_count(df_filtered, schema, line, cols)
    
    if not table_exist:
        create_tab(session, tab_name, schema)

    inserting(session, tab_name, schema, line)
    

1


In [62]:
# CHECK FOR NULL VALUE FUNCTION ****************
def null_count(df_filtered, schema, line, cols):
    
    for col_name in cols: 
        a = df_filtered.select(
            count(when(col(col_name).is_null(), 1)).alias(f"NULL_COUNT")
        ).collect()

        if len(line) == 1:
            line.append("'NULL_COUNT'")
            line.append("NULL")
        
        line.append(str(a[0][0]))
        schema.append(col_name)

    return line, schema

In [61]:
# CREATE TABLE IF NOT EXISTS *************************************************************************************
def create_tab(session, tab_name, schema):
    col_dtypes = []
        
    for i in schema:
        if i == 'SOURCE_FILE' or i == 'CHECK_TYPE':
            col_dtypes.append(i + " STRING")
        else:
            col_dtypes.append(i + " NUMERIC")
        
    query_create = f"""CREATE TABLE IF NOT EXISTS HOUSING_PRICE_PROJECT.STAGING_LAYER.{tab_name} ({", ".join(col_dtypes)});"""   
    session.sql(query_create).collect()

In [60]:
# INSERTING ROW WITH MISSING VALUE IF NOT EXISTS *****************************************************************

def inserting(session, tab_name, schema, line):
    
    report_tab = session.table(f"HOUSING_PRICE_PROJECT.STAGING_LAYER.{tab_name}")
    
    query_insert = f"""INSERT INTO HOUSING_PRICE_PROJECT.STAGING_LAYER.{tab_name} 
    ({", ".join(schema)})
    VALUES
    ({", ".join(line)});"""
    
    session.sql(query_insert).collect()

In [69]:
session.sql("""SELECT * FROM HOUSING_PRICE_PROJECT.STAGING_LAYER.TEST_TAB2""").to_pandas().head()

,SOURCE_FILE,CHECK_TYPE,OTHER,property_id,city,locality,locality_tier,property_type,bhk,bathrooms,...,power_backup,lift_available,maintenance_fee_monthly,distance_to_city_center_km,distance_to_metro_km,nearby_schools,nearby_hospitals,transaction_type,price_in_lakhs,price_category
0,HousingPricePredictionDataset_train.csv,NULL_COUNT,NaN,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Dropping duplicates and rows with too many NULLs

In [33]:
cols = df_snowpark.columns

iffs = """"""

for i in cols:
    iffs += f""" + IFF({i} IS NULL, 1, 0)"""

iffs = iffs[3:]
iffs

'IFF("property_id" IS NULL, 1, 0) + IFF("city" IS NULL, 1, 0) + IFF("locality" IS NULL, 1, 0) + IFF("locality_tier" IS NULL, 1, 0) + IFF("property_type" IS NULL, 1, 0) + IFF("bhk" IS NULL, 1, 0) + IFF("bathrooms" IS NULL, 1, 0) + IFF("balconies" IS NULL, 1, 0) + IFF("built_up_area" IS NULL, 1, 0) + IFF("carpet_area" IS NULL, 1, 0) + IFF("floor_number" IS NULL, 1, 0) + IFF("total_floors" IS NULL, 1, 0) + IFF("floor_category" IS NULL, 1, 0) + IFF("facing" IS NULL, 1, 0) + IFF("furnishing_status" IS NULL, 1, 0) + IFF("property_age" IS NULL, 1, 0) + IFF("parking_spaces" IS NULL, 1, 0) + IFF("security_score" IS NULL, 1, 0) + IFF("gym_available" IS NULL, 1, 0) + IFF("swimming_pool" IS NULL, 1, 0) + IFF("power_backup" IS NULL, 1, 0) + IFF("lift_available" IS NULL, 1, 0) + IFF("maintenance_fee_monthly" IS NULL, 1, 0) + IFF("distance_to_city_center_km" IS NULL, 1, 0) + IFF("distance_to_metro_km" IS NULL, 1, 0) + IFF("nearby_schools" IS NULL, 1, 0) + IFF("nearby_hospitals" IS NULL, 1, 0) + IFF("

In [55]:
sql_query = f"""SELECT DISTINCT * FROM HOUSING_PRICE_PROJECT.STAGING_LAYER.RAW_DATA\nWHERE {iffs} < 4 AND "price_in_lakhs" IS NOT NULL"""

In [53]:
ddl_query = f"""CREATE DYNAMIC TABLE IF NOT EXISTS HOUSING_PRICE_PROJECT.CLEANING_LAYER.CLEAN_DATA
TARGET_LAG = '1 day'
WAREHOUSE = housepriceproject_wh1
AS
{sql_query}
"""
print(ddl_query)

CREATE DYNAMIC TABLE IF NOT EXISTS HOUSING_PRICE_PROJECT.CLEANING_LAYER.CLEAN_DATA
TARGET_LAG = '1 day'
WAREHOUSE = housepriceproject_wh1
AS
SELECT DISTINCT * FROM HOUSING_PRICE_PROJECT.STAGING_LAYER.RAW_DATA
WHERE IFF("property_id" IS NULL, 1, 0) + IFF("city" IS NULL, 1, 0) + IFF("locality" IS NULL, 1, 0) + IFF("locality_tier" IS NULL, 1, 0) + IFF("property_type" IS NULL, 1, 0) + IFF("bhk" IS NULL, 1, 0) + IFF("bathrooms" IS NULL, 1, 0) + IFF("balconies" IS NULL, 1, 0) + IFF("built_up_area" IS NULL, 1, 0) + IFF("carpet_area" IS NULL, 1, 0) + IFF("floor_number" IS NULL, 1, 0) + IFF("total_floors" IS NULL, 1, 0) + IFF("floor_category" IS NULL, 1, 0) + IFF("facing" IS NULL, 1, 0) + IFF("furnishing_status" IS NULL, 1, 0) + IFF("property_age" IS NULL, 1, 0) + IFF("parking_spaces" IS NULL, 1, 0) + IFF("security_score" IS NULL, 1, 0) + IFF("gym_available" IS NULL, 1, 0) + IFF("swimming_pool" IS NULL, 1, 0) + IFF("power_backup" IS NULL, 1, 0) + IFF("lift_available" IS NULL, 1, 0) + IFF("maint

In [54]:
session.sql(ddl_query).collect()

[Row(status='Dynamic table CLEAN_DATA successfully created.')]